# Toolthread — BFCL multi-turn eval (Lightning AI)

Thin runner: clone the repo, install the pinned env, run the **full official multi-turn** BFCL eval against a base-model candidate, log official scores to W&B.

Set `HF_TOKEN` and `WANDB_API_KEY` in Lightning's environment/secrets before running. Lightning Studios with Ampere+ GPUs (L4 / A10G / A100) can A/B the faster **sglang** backend; vLLM is the safe default.

In [ ]:
import os, subprocess
REPO = "https://github.com/aniketqxp/toolthread.git"
if not os.path.isdir("toolthread"):
    subprocess.run(["git", "clone", REPO], check=True)
os.chdir("toolthread")
print("cwd:", os.getcwd())

In [ ]:
# Secrets come from Lightning's environment. Fail early if missing.
import os
for k in ("HF_TOKEN", "WANDB_API_KEY"):
    assert os.environ.get(k), f"Set {k} in Lightning's environment/secrets before running."
os.environ.setdefault("BFCL_PROJECT_ROOT", os.path.abspath("bfcl_runs"))
print("BFCL_PROJECT_ROOT =", os.environ["BFCL_PROJECT_ROOT"])

In [ ]:
# Pinned stable layer:
!pip install -q -r requirements.txt
# GPU serving layer via the official extra. vLLM default (works on every GPU):
!pip install -q "bfcl-eval[oss_eval_vllm,wandb]==2026.3.23"
# Ampere+ fast path — uncomment to A/B sglang on L4 / A10G / A100:
# !pip install -q "bfcl-eval[oss_eval_sglang,wandb]==2026.3.23"
# Make src/ importable:
!pip install -q -e . --no-deps

In [ ]:
# Baseline candidate A (Qwen3-1.7B), full multi-turn, vLLM.
!python scripts/run_bfcl_eval.py \
  --model-config configs/model/qwen3-1.7b.yaml \
  --eval-config  configs/eval/bfcl_multi_turn.yaml \
  --backend vllm

### Backend A/B (Ampere+ only)
On L4 / A10G / A100, re-run the cell above with `--backend sglang` (after installing the sglang extra) to compare. The harness records the backend in the W&B run config and the run manifest, so scores stay attributable.